# Distribuições flexíveis: vento e temperatura

Execute as células de cima para baixo. Quando aparecer o botão de upload,
selecione **`estacao_A001_diario.csv`**.

Vamos ajustar a GLD-FKML e compará-la com distribuições clássicas.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wmpjrufg/distribuicoes-flexiveis/blob/main/notebooks/distribuicoes_flexiveis_pt.ipynb)

In [ ]:
%pip install -q pyglam==1.1.0

import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pyglam import GlamFKML
from google.colab import files

## 1. Enviar e ler os dados

O CSV contém as colunas `data`, `vento_ms` e `temperatura_c`.

In [ ]:
uploaded = files.upload()
df = pd.read_csv(io.BytesIO(next(iter(uploaded.values()))))
df['data'] = pd.to_datetime(df['data'])
df.head()

## 2. Uma função para comparar os ajustes

O gráfico mostra o histograma e as densidades. A tabela mostra a distância
de Kolmogorov–Smirnov (`Dn`) e os desvios dos quantis de 5%, 50% e 95%.
Desvio = quantil do modelo − quantil dos dados. Quanto mais perto de zero, melhor.
Usamos `Dn` apenas como medida descritiva, sem teste de hipótese.

In [ ]:
def compare(x, models, xlabel):
    p = np.array([0.05, 0.50, 0.95])
    empirical = np.quantile(x, p)
    grid = np.linspace(x.min(), x.max(), 400)
    rows = []
    plt.figure(figsize=(7, 4))
    plt.hist(x, bins=30, density=True, alpha=0.3, color='gray')
    for name, model in models.items():
        plt.plot(grid, model.pdf(grid), label=name)
        delta = model.ppf(p) - empirical
        rows.append([name, stats.ks_1samp(x, model.cdf).statistic, *delta])
    plt.xlabel(xlabel)
    plt.ylabel('Densidade')
    plt.legend()
    plt.show()
    return pd.DataFrame(rows, columns=['Modelo', 'Dn', 'Δ5%', 'Δ50%', 'Δ95%']).round(3)

## 3. Velocidade média diária do vento

Usamos os valores disponíveis, sem preencher as falhas. A GLD é ajustada
pelo método dos momentos; Rayleigh e Weibull, por máxima verossimilhança,
com a origem fixada em zero.

In [ ]:
wind = df['vento_ms'].dropna().to_numpy()
fit_wind = GlamFKML().fit_lambdas(wind, n_starts=15, seed=0)
gld_wind = GlamFKML(*fit_wind.x)
_, rayleigh_scale = stats.rayleigh.fit(wind, floc=0)
shape, _, scale = stats.weibull_min.fit(wind, floc=0)

print('n =', len(wind))
print('λ =', fit_wind.x.round(4))
compare(wind, {
    'Rayleigh': stats.rayleigh(scale=rayleigh_scale),
    'Weibull': stats.weibull_min(shape, scale=scale),
    'GLD-FKML': gld_wind,
}, 'Velocidade do vento (m/s)')

### O suporte também importa

Um bom ajuste dos momentos não garante que o modelo inclua todos os dados.
Vamos contar quantos dias ficam fora dos limites da GLD ajustada.

In [ ]:
l1, l2, l3, l4 = fit_wind.x
lower = l1 - 1 / (l2 * l3) if l3 > 0 else -np.inf
upper = l1 + 1 / (l2 * l4) if l4 > 0 else np.inf
outside = np.sum((wind < lower) | (wind > upper))
print(f"Suporte: {lower:.3f} a {upper:.3f} m/s")
print(f"Dias fora do suporte: {outside} ({100 * outside / len(wind):.2f}%)")

## 4. Temperatura média diária de julho

Selecionamos julho para reduzir a mistura entre estações do ano.
Comparamos a GLD-FKML com a distribuição normal.

In [ ]:
temperature = df.loc[df['data'].dt.month == 7, 'temperatura_c'].dropna().to_numpy()
fit_temperature = GlamFKML().fit_lambdas(temperature, n_starts=15, seed=0)
gld_temperature = GlamFKML(*fit_temperature.x)
mean, std = stats.norm.fit(temperature)

print('n =', len(temperature))
print('λ =', fit_temperature.x.round(4))
compare(temperature, {
    'Normal': stats.norm(mean, std),
    'GLD-FKML': gld_temperature,
}, 'Temperatura (°C)')

## Para pensar

- Qual modelo tem os menores desvios de quantis no vento?
- O que significa a GLD excluir alguns dias do seu suporte?
- A melhora no ajuste da temperatura justifica usar quatro parâmetros em vez de dois?

Estes são ajustes descritivos. Os dados diários têm dependência temporal,
e a incerteza amostral não é avaliada neste exemplo curto.